# Self-Supervised and Contrastive Learning — Exercises

Companion to the note [Self-Supervised and Contrastive Learning](Self-Supervised%20and%20Contrastive%20Learning.md).

Practise the InfoNCE / NT-Xent loss by hand and in code (including its gradient through L2 normalisation), the role of the temperature, the CLIP loss, momentum encoders, collapse diagnostics (alignment and uniformity) and linear probing. Everything is NumPy, checked against sklearn, brute-force loops and finite differences.

**17 exercises** · 🧠 Concept ×3 · ✍️ By hand ×5 · 💻 Code ×9

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import softmax as sp_softmax, logsumexp
from scipy.spatial.distance import pdist
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import log_loss
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(4)

def num_grad(f, x, eps=1e-6):
    """Central finite-difference gradient of a scalar function f() w.r.t. array x (modified in place, restored)."""
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * eps)
    return g

## Part A · Concepts

### Exercise 1 · Name the pretext task
*🧠 Concept · ★☆☆*

Classify each method into the note's families (contrastive, non-contrastive, masked modeling, autoregressive) and state its pretext task:
SimCLR, BYOL, MAE, GPT, CLIP, DINO, BERT, MoCo.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Ask: are there negatives? Is something hidden and reconstructed? Is the next token predicted?

</details>

<details>
<summary><b>✅ Solution</b></summary>

- **Contrastive**: SimCLR (two augmented views of an image are positives, other images in the batch negatives), MoCo (same, negatives from a queue,
  keys from a momentum encoder), CLIP (matching image–text pairs vs all other pairs in the batch).
- **Non-contrastive**: BYOL (online network predicts a momentum target network's embedding of another view), DINO (self-distillation with centering and sharpening).
- **Masked modeling**: MAE (reconstruct ~75% masked image patches), BERT (predict masked tokens; [[Transformers]]).
- **Autoregressive**: GPT (next-token prediction).

</details>

### Exercise 2 · Collapse and how each family avoids it
*🧠 Concept · ★★☆*

The note calls **collapse** (all embeddings equal) the failure mode to watch.

1. Why does a loss that only pulls positive pairs together have a trivial minimiser?
2. How do negatives in InfoNCE prevent it?
3. BYOL/SimSiam have no negatives. Which ingredients prevent collapse there?
4. Name a cheap diagnostic you can log during training.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A constant encoder makes every positive pair identical.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $f(x)\equiv c$ makes every positive distance zero, the global minimum, but carries no information.
2. The denominator penalises similarity to negatives: a collapsed encoder makes all similarities equal, giving loss $\log(2N-1)$, far from optimal.
   InfoNCE asymptotically optimises **alignment** + **uniformity** on the sphere (Exercise 14).
3. An asymmetric **predictor** head on the online branch, **stop-gradient** on the target branch, and (BYOL) a **momentum/EMA** target encoder (Exercise 17).
   DINO additionally uses centering and sharpening of the teacher outputs.
4. The per-dimension standard deviation of L2-normalised embeddings (≈ $1/\sqrt d$ when healthy, → 0 when collapsed), the uniformity metric, or the effective rank of the embedding covariance.

</details>

### Exercise 3 · Linear probe, projection head and augmentations
*🧠 Concept · ★★☆*

1. What is a linear probe, and why is it a fairer measure of representation quality than full fine-tuning?
2. SimCLR computes the loss on $z = g(h)$ (projection head) but uses $h$ for downstream tasks. Why throw $g$ away?
3. Why are *crop + colour jitter* so important? What would the network learn with crops only?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The contrastive loss makes $z$ invariant to the augmentations. Is that invariance always useful downstream?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Freeze the encoder and train only a linear classifier on its features. It measures how linearly separable the learned features are,
   without the encoder adapting to labels (fine-tuning can hide a weak representation and depends on many hyperparameters).
2. The loss forces $z$ to be **invariant** to the augmentations and discards information (colour, orientation) that can matter downstream.
   $h$, one layer before, retains more; empirically probes on $h$ are 10%+ better.
3. Random crops of the same image share colour histograms, so with crops only the network can solve the task by matching colour statistics
   (a shortcut). Colour jitter removes that shortcut and forces it to use shape/semantics.

</details>

## Part B · By hand

### Exercise 4 · InfoNCE for one anchor
*✍️ By hand · ★☆☆*

An anchor has cosine similarity $0.8$ with its positive and $0.2, -0.1$ with two negatives; temperature $\tau = 0.5$.
Using the note's formula (denominator over all $k\ne i$, which includes the positive), compute $\ell_i$.

In [ ]:
loss_hand = None

check('InfoNCE', loss_hand, -np.log(sp_softmax(np.array([0.8, 0.2, -0.1]) / 0.5)[0]), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Logits $s/\tau = (1.6, 0.4, -0.2)$. $\ell = -1.6 + \log(e^{1.6}+e^{0.4}+e^{-0.2})$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$e^{1.6} = 4.953$, $e^{0.4} = 1.492$, $e^{-0.2} = 0.819$; sum $7.264$. $\ell = -\log(4.953/7.264) = -\log 0.6819 = 0.3829$.
It is exactly a cross-entropy where the "correct class" is the positive.

```python
loss_hand = -1.6 + np.log(np.exp(1.6) + np.exp(0.4) + np.exp(-0.2))
```

</details>

### Exercise 5 · Loss of a collapsed encoder
*✍️ By hand · ★☆☆*

In SimCLR a batch of $N$ images gives $2N$ views; each anchor has 1 positive and $2N-2$ negatives. If the encoder collapses, all similarities are equal.
What is the NT-Xent loss? Evaluate for $N = 256$.

In [ ]:
loss_collapsed = None

_s = np.zeros(2 * 256 - 1)
check('collapsed loss', loss_collapsed, -np.log(sp_softmax(_s)[0]), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

All $2N-1$ terms in the denominator are equal to the numerator.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\ell = -\log\frac{e^{c}}{(2N-1)e^c} = \log(2N-1) = \log 511 \approx 6.236$. Seeing this value (independent of $\tau$) in your training log means collapse;
a randomly initialised encoder also starts near it.

```python
loss_collapsed = np.log(511)
```

</details>

### Exercise 6 · Gradient w.r.t. the similarities
*✍️ By hand · ★★☆*

Let $p_k = \mathrm{softmax}(s/\tau)_k$ over the denominator set. Show that
$\frac{\partial\ell}{\partial s_k} = \frac{1}{\tau}(p_k - \mathbb 1[k=\text{pos}])$.
For Exercise 4's numbers compute the gradient w.r.t. the positive similarity and the first negative's similarity.
How does $\tau$ affect which negatives receive most gradient?

In [ ]:
g_pos = None
g_neg1 = None

_s = np.array([0.8, 0.2, -0.1]); _f = lambda s: -np.log(sp_softmax(s / 0.5)[0]); _h = 1e-6
check('d loss / d s_pos', g_pos, (_f(_s + [_h, 0, 0]) - _f(_s - [_h, 0, 0])) / (2 * _h), tol=1e-5)
check('d loss / d s_neg1', g_neg1, (_f(_s + [0, _h, 0]) - _f(_s - [0, _h, 0])) / (2 * _h), tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

$\ell = -s_{pos}/\tau + \log\sum_k e^{s_k/\tau}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Differentiating the log-sum-exp gives $p_k/\tau$; the first term contributes $-1/\tau$ for the positive.
Here $p = (0.6819, 0.2054, 0.1127)$: $\partial\ell/\partial s_{pos} = (0.6819-1)/0.5 = -0.636$, $\partial\ell/\partial s_{neg_1} = 0.2054/0.5 = 0.411$.
Negatives are pushed with weight $\propto p_k$: a **small** $\tau$ sharpens $p$ so the hardest negatives (most similar) dominate (hard-negative mining);
a large $\tau$ spreads the gradient uniformly. Too small a $\tau$ also pushes apart semantically similar "false negatives".

```python
_p = sp_softmax(np.array([0.8, 0.2, -0.1]) / 0.5)
g_pos = (_p[0] - 1) / 0.5
g_neg1 = _p[1] / 0.5
```

</details>

### Exercise 7 · Cosine similarity vs Euclidean distance
*✍️ By hand · ★☆☆*

Show that for unit vectors $\lVert u-v\rVert^2 = 2 - 2\cos(u,v)$. Then for $u = (0.6, 0.8)$, $v = (1, 0)$ compute $\cos(u,v)$ and $\lVert u-v\rVert^2$.
Why do contrastive methods L2-normalise embeddings before computing similarities?

In [ ]:
cos_uv = None
sqdist_uv = None

_u, _v = np.array([0.6, 0.8]), np.array([1., 0])
check('cosine', cos_uv, cosine_similarity([_u], [_v])[0, 0]); check('squared distance', sqdist_uv, np.sum((_u - _v) ** 2))

<details>
<summary><b>💡 Hint</b></summary>

Expand $\lVert u-v\rVert^2 = \lVert u\rVert^2 + \lVert v\rVert^2 - 2u^\top v$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With $\lVert u\rVert=\lVert v\rVert=1$: $\lVert u-v\rVert^2 = 2-2u^\top v$. Here $\cos = 0.6$, distance² $= 0.8$.
Normalising puts embeddings on the unit sphere, so the loss cannot be reduced by just scaling up norms; similarity is bounded in $[-1,1]$ and $\tau$ alone controls the logit scale.

```python
cos_uv = 0.6
sqdist_uv = 0.8
```

</details>

### Exercise 8 · InfoNCE as a mutual-information bound
*✍️ By hand · ★★☆*

For InfoNCE with one positive among $N$ candidates, $I(x; y) \ge \log N - \mathcal L_{\text{InfoNCE}}$ (van den Oord et al., 2018).
(a) A model trained with $N = 1024$ candidates reaches $\mathcal L = 3.0$ nats. What lower bound on the MI does this certify?
(b) What is the largest MI value this estimator can ever certify with $N = 1024$, and what does that imply for batch size?

In [ ]:
mi_bound = None
mi_max = None

check('MI lower bound', mi_bound, np.log(1024) - 3.0, tol=1e-4)
check('max certifiable MI', mi_max, np.log(1024), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

The loss is non-negative.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\log 1024 - 3 = 6.931 - 3 = 3.931$ nats. (b) Since $\mathcal L \ge 0$ the bound is at most $\log N = 6.931$ nats ($= 10$ bits).
To certify more information you need more negatives: one theoretical motivation for SimCLR's large batches and MoCo's queue.

```python
mi_bound = np.log(1024) - 3.0
mi_max = np.log(1024)
```

</details>

## Part C · Code from scratch

### Exercise 9 · Cosine similarity matrix
*💻 Code · ★☆☆*

Implement `cos_sim(A, B)` returning the matrix of cosine similarities between rows of `A` and rows of `B` (L2-normalise, then a matrix product).

In [ ]:
def l2n(Z):
    return None

def cos_sim(A, B):
    return None

A_c, B_c = rng.normal(size=(5, 8)), rng.normal(size=(7, 8))

check('cosine similarity matrix', cos_sim(A_c, B_c), cosine_similarity(A_c, B_c))

<details>
<summary><b>💡 Hint</b></summary>

`Z / np.linalg.norm(Z, axis=1, keepdims=True)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

After normalisation, the Gram matrix is the cosine-similarity matrix. This is one GEMM, which is why large batches are cheap on GPUs.

```python
def l2n(Z):
    return Z / np.linalg.norm(Z, axis=1, keepdims=True)

def cos_sim(A, B):
    return l2n(A) @ l2n(B).T

A_c, B_c = rng.normal(size=(5, 8)), rng.normal(size=(7, 8))
```

</details>

### Exercise 10 · NT-Xent loss for a batch
*💻 Code · ★★☆*

Implement `nt_xent(z1, z2, tau)`: `z1[i]` and `z2[i]` are two views of image $i$ (shape `(N, d)`). Stack them into $2N$ embeddings,
normalise, compute all similarities, **exclude self-similarity** ($k\ne i$), and average $\ell_i$ over all $2N$ anchors (the positive of
row $i$ is row $i+N$ and vice versa). Vectorise it: no Python loop over anchors.

In [ ]:
def nt_xent(z1, z2, tau=0.5):
    return None

z1_b, z2_b = rng.normal(size=(6, 4)), rng.normal(size=(6, 4))
loss_b = nt_xent(z1_b, z2_b, 0.5)
eye_views = nt_xent(np.eye(6) * 5, np.eye(6) * 5, tau=0.05)

def _ref(z1, z2, tau):   # brute-force double loop
    Z = np.r_[z1, z2]; Z = Z / np.linalg.norm(Z, axis=1, keepdims=True); n2 = len(Z); N = n2 // 2; L = 0
    for i in range(n2):
        j = (i + N) % n2
        den = sum(np.exp(Z[i] @ Z[k] / tau) for k in range(n2) if k != i)
        L += -np.log(np.exp(Z[i] @ Z[j] / tau) / den)
    return L / n2
check('NT-Xent = brute force', loss_b, _ref(z1_b, z2_b, 0.5))
check('perfect, orthogonal views -> loss ~ 0', None if eye_views is None else eye_views < 1e-6, True)

<details>
<summary><b>💡 Hint 1</b></summary>

`S = Z @ Z.T / tau`, then `np.fill_diagonal(S, -np.inf)` removes $k=i$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Positives: `pos = np.r_[np.arange(N, 2*N), np.arange(N)]`; loss $= -S[i, pos_i] + \mathrm{logsumexp}(S[i])$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Setting the diagonal to $-\infty$ removes self-similarity (which would otherwise be the largest logit, $1/\tau$). Using `logsumexp` keeps it stable even for $\tau = 0.05$.
With orthogonal images and identical views, positives have logit $20$ and negatives $0$, so the loss is $\approx \log(1 + 10e^{-20}) \approx 0$.

```python
def nt_xent(z1, z2, tau=0.5):
    Z = l2n(np.r_[z1, z2]); N = len(z1)
    S = Z @ Z.T / tau
    np.fill_diagonal(S, -np.inf)
    pos = np.r_[np.arange(N, 2 * N), np.arange(N)]
    return np.mean(-S[np.arange(2 * N), pos] + logsumexp(S, axis=1))

z1_b, z2_b = rng.normal(size=(6, 4)), rng.normal(size=(6, 4))
loss_b = nt_xent(z1_b, z2_b, 0.5)
eye_views = nt_xent(np.eye(6) * 5, np.eye(6) * 5, tau=0.05)
```

</details>

### Exercise 11 · Backpropagating NT-Xent through the normalisation
*💻 Code · ★★★*

Implement `nt_xent_grad(z1, z2, tau)` returning $\partial\mathcal L/\partial z_1$ and $\partial\mathcal L/\partial z_2$ (the **unnormalised** embeddings).
Steps: gradient w.r.t. the logit matrix $S$ (Exercise 6), then w.r.t. the normalised $\hat Z$, then through $\hat z = z/\lVert z\rVert$.

In [ ]:
def nt_xent_grad(z1, z2, tau=0.5):
    return None, None

g1_b, g2_b = nt_xent_grad(z1_b, z2_b, 0.5)

def _L():
    Z = np.r_[z1_b, z2_b]; Z = Z / np.linalg.norm(Z, axis=1, keepdims=True); N = len(z1_b)
    S = Z @ Z.T / 0.5; np.fill_diagonal(S, -np.inf); pos = np.r_[np.arange(N, 2 * N), np.arange(N)]
    return np.mean(-S[np.arange(2 * N), pos] + logsumexp(S, axis=1))
check('dL/dz1 (finite differences)', g1_b, num_grad(_L, z1_b), tol=1e-5)
check('dL/dz2 (finite differences)', g2_b, num_grad(_L, z2_b), tol=1e-5)

<details>
<summary><b>💡 Hint 1</b></summary>

$G = \partial\mathcal L/\partial S = (P - Y)/(2N)$ with $P$ the row-softmax (diagonal 0) and $Y$ the one-hot positives.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$S = \hat Z\hat Z^\top/\tau \Rightarrow \partial\mathcal L/\partial\hat Z = (G+G^\top)\hat Z/\tau$. Through the normalisation: $\partial\mathcal L/\partial z = (g - \hat z\,(\hat z^\top g))/\lVert z\rVert$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The Jacobian of $z\mapsto z/\lVert z\rVert$ is $(I-\hat z\hat z^\top)/\lVert z\rVert$: it removes the radial component of the gradient, so updates only rotate
embeddings on the sphere (and large-norm embeddings get smaller gradients). The positive pulls the anchor towards its partner; every negative pushes with weight $p_k$.

```python
def nt_xent_grad(z1, z2, tau=0.5):
    Zr = np.r_[z1, z2]; nrm = np.linalg.norm(Zr, axis=1, keepdims=True); Z = Zr / nrm
    N = len(z1); n2 = 2 * N
    S = Z @ Z.T / tau; np.fill_diagonal(S, -np.inf)
    P = np.exp(S - logsumexp(S, axis=1, keepdims=True))
    Y = np.zeros((n2, n2)); Y[np.arange(n2), np.r_[np.arange(N, n2), np.arange(N)]] = 1
    G = (P - Y) / n2
    gZ = (G + G.T) @ Z / tau
    gz = (gZ - Z * np.sum(Z * gZ, axis=1, keepdims=True)) / nrm
    return gz[:N], gz[N:]

g1_b, g2_b = nt_xent_grad(z1_b, z2_b, 0.5)
```

</details>

### Exercise 12 · Augmentations for two views
*💻 Code · ★☆☆*

On $8\times8$ digit images implement `hflip(img)`, `shift_crop(img, dy, dx, pad=1)` (zero-pad by `pad`, then cut the $8\times8$ window starting at `(dy, dx)`)
and `two_views(img, rng)` that applies a random flip, a random crop offset in $\{0,1,2\}^2$ and Gaussian noise ($\sigma=0.5$) independently twice.
Is horizontal flipping a sensible augmentation for digits?

In [ ]:
digits = load_digits()
img0 = digits.images[0]

def hflip(img):
    return None

def shift_crop(img, dy, dx, pad=1):
    return None

def two_views(img, rng):
    return None, None

v1, v2 = two_views(img0, rng)

check('flip twice = identity', None if hflip(img0) is None else hflip(hflip(img0)), img0)
check('centred crop = identity', shift_crop(img0, 1, 1), img0)
check('crop (0,0) shifts content down-right', shift_crop(img0, 0, 0), np.pad(img0, 1)[:8, :8])
check('views have the right shape', None if v1 is None else (v1.shape, v2.shape), ((8, 8), (8, 8)))

<details>
<summary><b>💡 Hint</b></summary>

`img[:, ::-1]`; `np.pad(img, pad)[dy:dy+8, dx:dx+8]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Flips are standard for natural images but questionable for digits/text: a flipped 2 or 7 is not a valid digit, and some digits become others
(augmentations must preserve the semantics you care about). Crops/shifts and noise are safe here.

```python
digits = load_digits()
img0 = digits.images[0]

def hflip(img):
    return img[:, ::-1]

def shift_crop(img, dy, dx, pad=1):
    H, W = img.shape
    return np.pad(img, pad)[dy:dy + H, dx:dx + W]

def two_views(img, rng):
    views = []
    for _ in range(2):
        v = hflip(img) if rng.random() < 0.5 else img
        v = shift_crop(v, *rng.integers(0, 3, size=2))
        views.append(v + 0.5 * rng.normal(size=v.shape))
    return views[0], views[1]

v1, v2 = two_views(img0, rng)
```

</details>

### Exercise 13 · Temperature controls the sharpness
*💻 Code · ★☆☆*

An anchor has similarity $0.9$ to its positive and 255 negatives with similarities drawn uniformly from $[-0.2, 0.7]$ (fixed below).
For $\tau\in\{0.05, 0.1, 0.5, 1.0\}$ compute the softmax distribution over the 256 candidates; store its entropy (nats) in `ent[tau]`
and the loss in `loss_tau[tau]`. Explain the trends.

In [ ]:
sims = np.r_[0.9, rng.uniform(-0.2, 0.7, size=255)]
taus = (0.05, 0.1, 0.5, 1.0)
ent = None       # dict
loss_tau = None  # dict

check('entropy increases with tau', None if ent is None else bool(np.all(np.diff([ent[t] for t in taus]) > 0)), True)
check('loss increases with tau here', None if loss_tau is None else bool(np.all(np.diff([loss_tau[t] for t in taus]) > 0)), True)
check('tau=1 entropy close to log(256)', None if ent is None else abs(ent[1.0] - np.log(256)) < 0.1, True)

<details>
<summary><b>💡 Hint</b></summary>

`p = sp_softmax(sims / tau)`; entropy $-\sum p\log p$; loss $-\log p_0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Small $\tau$ makes the softmax peaky (low entropy): with the positive clearly most similar the loss is small, and gradients concentrate on the hardest negatives.
With $\tau = 1$ the logits span only $[-0.2, 0.9]$, so the distribution is nearly uniform (entropy ≈ $\log 256 = 5.55$) and the loss stays high (≈ 4.9) even though the positive is the most similar candidate.
That is why SimCLR uses $\tau\approx 0.1$–$0.5$ and CLIP learns $\tau$.

```python
sims = np.r_[0.9, rng.uniform(-0.2, 0.7, size=255)]
taus = (0.05, 0.1, 0.5, 1.0)
ent, loss_tau = {}, {}
for t in taus:
    p = sp_softmax(sims / t)
    ent[t] = -np.sum(p * np.log(p)); loss_tau[t] = -np.log(p[0])
print(ent, loss_tau)
```

</details>

### Exercise 14 · Alignment and uniformity
*💻 Code · ★★☆*

Wang & Isola (2020) showed contrastive learning optimises two properties of normalised embeddings $f$:
$\mathcal L_{align} = \mathbb E_{\text{pos}}\lVert f(x)-f(y)\rVert^2$ and
$\mathcal L_{unif} = \log\mathbb E_{x,y\text{ i.i.d.}} e^{-2\lVert f(x)-f(y)\rVert^2}$ (over all distinct pairs).
Implement both and evaluate `uniformity` for (a) random points on the sphere in $\mathbb R^{16}$, (b) a collapsed embedding.

In [ ]:
def alignment(F1, F2):
    return None

def uniformity(F):
    return None

F_sphere = rng.normal(size=(500, 16)); F_sphere /= np.linalg.norm(F_sphere, axis=1, keepdims=True)
F_collapsed = np.tile(F_sphere[:1], (500, 1))
u_sphere = uniformity(F_sphere)
u_collapsed = uniformity(F_collapsed)

check('uniformity = log mean exp(-2 d^2) over pairs', u_sphere, np.log(np.mean(np.exp(-2 * pdist(F_sphere, 'sqeuclidean')))))
check('collapsed: uniformity = 0 (worst)', u_collapsed, 0.0)
check('alignment of identical views = 0', alignment(F_sphere, F_sphere), 0.0)
check('alignment = 2 - 2 cos for unit vectors', alignment(F_sphere[:250], F_sphere[250:]), np.mean(2 - 2 * np.sum(F_sphere[:250] * F_sphere[250:], 1)))

<details>
<summary><b>💡 Hint</b></summary>

Pairwise squared distances: `pdist(F, 'sqeuclidean')` or via the Gram matrix ($2-2G$) taking the upper triangle with `np.triu_indices(n, 1)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Lower is better for both. A collapsed encoder has perfect alignment (0) but the worst uniformity (0); spreading points uniformly on the sphere gives
$\approx -3.5$ here (approaching $-4$ as the dimension grows, since then $\lVert x-y\rVert^2\approx 2$ for almost all pairs). InfoNCE's numerator drives alignment, its denominator uniformity.

```python
def alignment(F1, F2):
    return np.mean(np.sum((F1 - F2) ** 2, axis=1))

def uniformity(F):
    G = F @ F.T
    iu = np.triu_indices(len(F), 1)
    d2 = np.maximum(2 - 2 * G[iu], 0)
    return np.log(np.mean(np.exp(-2 * d2)))

F_sphere = rng.normal(size=(500, 16)); F_sphere /= np.linalg.norm(F_sphere, axis=1, keepdims=True)
F_collapsed = np.tile(F_sphere[:1], (500, 1))
u_sphere = uniformity(F_sphere)
u_collapsed = uniformity(F_collapsed)
print(u_sphere, u_collapsed)
```

</details>

### Exercise 15 · Linear probing
*💻 Code · ★★☆*

Simulate three frozen "encoders" for the digits dataset (all 16-dimensional): (a) PCA-16 features, (b) a random linear projection + ReLU,
(c) a collapsed encoder (the same vector for every image plus $10^{-3}$ noise). Train a linear probe (`LogisticRegression(max_iter=2000)`) on a
70/30 split (`random_state=0`, stratified) and store the test accuracies in `probe_acc` (keys `'pca', 'random', 'collapsed'`).

In [ ]:
Xd, yd = load_digits(return_X_y=True)
Xd = Xd / 16.0
probe_acc = None

check('PCA features are linearly separable (> 0.85)', None if probe_acc is None else probe_acc['pca'] > 0.85, True)
check('collapsed encoder is at chance (< 0.2)', None if probe_acc is None else probe_acc['collapsed'] < 0.2, True)

<details>
<summary><b>💡 Hint</b></summary>

Fit PCA on the training split only. `train_test_split(F, yd, test_size=0.3, random_state=0, stratify=yd)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

PCA features reach ~0.9+, a random projection is usually somewhat worse, and the collapsed encoder sits at chance (~0.1): a linear probe immediately exposes collapse.
In SSL papers the probe is run on the frozen backbone (before the projection head) after pretraining.

```python
Xd, yd = load_digits(return_X_y=True)
Xd = Xd / 16.0
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.3, random_state=0, stratify=yd)
W_rand = rng.normal(size=(64, 16))
encoders = {
    'pca': (lambda P: (lambda X: P.transform(X)))(PCA(16, random_state=0).fit(Xtr)),
    'random': lambda X: np.maximum(0, X @ W_rand),
    'collapsed': lambda X: np.ones((len(X), 16)) + 1e-3 * rng.normal(size=(len(X), 16)),
}
probe_acc = {}
for name, enc in encoders.items():
    clf = LogisticRegression(max_iter=2000).fit(enc(Xtr), ytr)
    probe_acc[name] = clf.score(enc(Xte), yte)
print(probe_acc)
```

</details>

### Exercise 16 · CLIP's symmetric contrastive loss
*💻 Code · ★★☆*

CLIP embeds $N$ images and their $N$ captions. With normalised embeddings, logits $L = \hat I\hat T^\top/\tau$; the target for row $i$ and column $i$ is the diagonal.
Implement `clip_loss(I, T, tau)` $= \tfrac12\big(\mathrm{CE}_{\text{rows}}(L) + \mathrm{CE}_{\text{cols}}(L)\big)$ (mean cross-entropy over the batch in each direction).
How does it differ from NT-Xent?

In [ ]:
def clip_loss(I, T, tau=0.07):
    return None

I_e, T_e = rng.normal(size=(8, 5)), rng.normal(size=(8, 5))
loss_clip = clip_loss(I_e, T_e, 0.5)

_L = cosine_similarity(I_e, T_e) / 0.5
_ce_rows = log_loss(np.arange(8), sp_softmax(_L, axis=1), labels=np.arange(8))
_ce_cols = log_loss(np.arange(8), sp_softmax(_L.T, axis=1), labels=np.arange(8))
check('CLIP loss', loss_clip, (_ce_rows + _ce_cols) / 2)

<details>
<summary><b>💡 Hint</b></summary>

Rows: `-mean(diag(log_softmax(L, axis=1)))`; columns: same with `axis=0`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Each image must pick its caption among $N$ captions (rows), and each caption its image among $N$ images (columns).
Unlike NT-Xent there are **two modalities**: negatives are only cross-modal (no image–image pairs), and there is no self-similarity to mask.
It is the same in-batch softmax as two-tower recommenders ([[Deep Learning Recommenders]]). CLIP learns $\tau$ (init 0.07).

```python
def clip_loss(I, T, tau=0.07):
    L = l2n(I) @ l2n(T).T / tau
    ce_rows = -np.mean(np.diag(L - logsumexp(L, axis=1, keepdims=True)))
    ce_cols = -np.mean(np.diag(L - logsumexp(L, axis=0, keepdims=True)))
    return (ce_rows + ce_cols) / 2

I_e, T_e = rng.normal(size=(8, 5)), rng.normal(size=(8, 5))
loss_clip = clip_loss(I_e, T_e, 0.5)
```

</details>

### Exercise 17 · Momentum (EMA) encoder
*💻 Code · ★☆☆*

MoCo and BYOL keep a *target* encoder updated as $\xi \leftarrow m\,\xi + (1-m)\,\theta$ instead of by gradients.
Implement `ema_update(xi, theta, m)`. Starting from $\xi_0 = 0$ with a **fixed** online $\theta$, run 100 steps with $m = 0.99$; store $\xi_{100}$ in `xi_100`.
Derive the closed form and say why a slowly moving target helps.

In [ ]:
def ema_update(xi, theta, m=0.99):
    return None

theta_on = rng.normal(size=5)
xi_100 = None

check('EMA after 100 steps = closed form', xi_100, theta_on * (1 - 0.99 ** 100))

<details>
<summary><b>💡 Hint</b></summary>

$\xi_k - \theta = m(\xi_{k-1} - \theta)$, so the gap shrinks geometrically.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\xi_k = \theta + m^k(\xi_0 - \theta)$, so here $\xi_{100} = (1 - 0.99^{100})\theta \approx 0.634\,\theta$: the target averages the last ~$1/(1-m) = 100$ online weights.
A slowly changing target gives **consistent** keys for MoCo's queue and stable regression targets for BYOL, which together with the predictor avoids collapse.

```python
def ema_update(xi, theta, m=0.99):
    return m * xi + (1 - m) * theta

theta_on = rng.normal(size=5)
xi = np.zeros(5)
for _ in range(100):
    xi = ema_update(xi, theta_on, 0.99)
xi_100 = xi
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Self-Supervised and Contrastive Learning](Self-Supervised%20and%20Contrastive%20Learning.md).*